# 🧠 Think-Vetor 1.5B: Treinamento Híbrido (SFT + GRPO-RL)

Este notebook foi projetado para rodar na nuvem (**Google Colab** ou **Vast.ai**) para treinar o **Think-Vetor 1.5B** utilizando **Supervised Fine-Tuning (SFT)** combinado com **Group Relative Policy Optimization (GRPO) Reinforcement Learning**.

## 🚀 Por que treinar na Nuvem?
Treinar modelos de 1.5B parâmetros com o pipeline GRPO-RL requer computação paralela de amostragem de grupo ($G=4$ completions por prompt). Fazer isso localmente em CPUs ou GPUs domésticas com pouca VRAM pode causar travamentos do sistema (OOM) ou demorar muito tempo. 

Na nuvem (ex: GPU T4/L4 do Colab ou RTX 3090/4090 do Vast.ai), o treinamento é acelerado e concluído de forma estável.

## 1. Configuração do Ambiente e Dependências

In [ ]:
# 1. Instalar bibliotecas necessárias para QLoRA e Treinamento de RL
!pip install -q transformers peft bitsandbytes accelerate trl safetensors
print("[SUCESSO] Dependências instaladas!")

In [ ]:
# 2. Configurar diretório de trabalho e clonar repositório se necessário
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    print("[INFO] Detectado subdiretório notebooks/. Movendo para a raiz...")
    %cd ..

if not os.path.exists("src"):
    print("[INFO] Clonando repositório...")
    !git clone https://github.com/MrJc01/crom-microllm-think-vetor.git
    %cd crom-microllm-think-vetor
else:
    print("[INFO] Já no diretório do projeto.")

sys.path.append(os.getcwd())
print(f"Diretório atual: {os.getcwd()}")

## 2. Geração do Dataset Sintético Conversacional

Antes de iniciar o treino, precisamos gerar os 500 diálogos multi-turn balanceados (Persona Defense, Lógica Relacional, Aritmética TV-DSL, e Restrições de Prompt).

In [ ]:
# Gerar o dataset conversacional
!python generate_synthetic_conversations.py

## 3. Execução do Treinamento Híbrido (1.5B SFT + GRPO-RL)

Dispararemos o script `train_hybrid_1b.py` especificando o modelo base de 1.5B parameters (`Qwen/Qwen2.5-1.5B-Instruct`). O treino alternará:
- **Época 0**: Supervised Fine-Tuning (SFT) para estabilizar a escrita de tags `<thought>` e comportamento de persona.
- **Épocas 1 e 2**: GRPO-RL com 5 funções de recompensa ativas (Persona, Formato CoT, Ativação TV-DSL, Resolução AST e Acurácia Final).

*Nota: O script utilizará automaticamente o otimizador economizador de memória `PagedAdamW8bit` e quantização de 4-bits (QLoRA) para caber com folga em qualquer GPU de 15GB+ VRAM.*

In [ ]:
# Iniciar o treinamento híbrido no modelo de 1.5B
!python train_hybrid_1b.py \
    --model_id "Qwen/Qwen2.5-1.5B-Instruct" \
    --epochs 3 \
    --switch_epoch 1 \
    --batch_size 2 \
    --lr 2e-5 \
    --out_dir "checkpoints/think_vetor_1b_hybrid_lora"

## 4. Bateria de Testes Automatizada (50 Casos / 250 Diálogos)

Avalie o modelo sintonizado contra desvio de persona (sycophancy), integridade relacional lógica e ativação determinística da TV-DSL.

In [ ]:
# Rodar a bateria de testes de robustez conversacional
!python run_multiturn_tests.py

## 5. Empacotamento e Download dos Adaptadores LoRA

Após a conclusão dos testes, podemos compactar a pasta de checkpoints do adaptador LoRA treinado para baixá-la e utilizá-la em nosso Playground Web local.

In [ ]:
# Compactar os adaptadores em formato zip
!zip -r think_vetor_1b_hybrid_lora.zip checkpoints/think_vetor_1b_hybrid_lora/
print("[INFO] Adaptadores LoRA compactados com sucesso em 'think_vetor_1b_hybrid_lora.zip'.")

In [ ]:
# (Apenas para Google Colab) Disparar o download automático para a sua máquina local
try:
    from google.colab import files
    files.download("think_vetor_1b_hybrid_lora.zip")
except ImportError:
    print("[INFO] Executando fora do Colab. Baixe o arquivo 'think_vetor_1b_hybrid_lora.zip' manualmente ou envie-o via SCP/FTP da sua instância Vast.ai.")